# Evidencia Evaluativa 1: obtener y preparar datos desde Hugging Face

**Integrantes:** Jaider Morales Bautista · Aleicer Vesga Rueda · Guillerno Leon Loaiza Mesa

En este notebook coordinamos las dos clases del proyecto:

1. `HFDatasetLoader` (`src/clases_po/hf_loader.py`): se conecta a Hugging Face y devuelve el dataset como DataFrame.
2. `Reportes` (`src/clases_po/reportes.py`): recibe un DataFrame y genera un resumen en `.txt`.

El flujo es: **cargar → transformar → resumir**. El notebook se ejecuta automáticamente en GitHub Actions (`.github/workflows/ea1.yml`), así que el reporte `.txt` siempre corresponde a la última ejecución.

**Dataset:** [`pointe77/credit-card-transaction`](https://huggingface.co/datasets/pointe77/credit-card-transaction), split `train`. Son transacciones de tarjeta de crédito simuladas, con datos del cliente, del comercio y una marca que dice si la transacción fue fraude.

## 1. Importar las clases

Buscamos la raíz del proyecto para poder importar las clases desde `src/clases_po`, sin importar desde qué carpeta se ejecute el notebook.

In [1]:
import sys 
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

/opt/hostedtoolcache/Python/3.12.15/x64/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Cargar el dataset desde Hugging Face

Usamos `max_rows=100` para que esta primera actividad sea liviana: solo necesitamos ver la estructura de los datos y probar una transformación. En la actividad 2 se usa una muestra mucho más grande para entrenar los modelos.

In [2]:
loader = HFDatasetLoader(
    dataset_name="pointe77/credit-card-transaction",
    split="train",
    max_rows=100,
    filename="datos/pointe77/credit-card-transaction.csv",
)

In [3]:
df = loader.load()
df.head(2)

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,merch_zipcode
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0,28705.0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0,NaN


In [4]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

Filas: 100
Columnas: 24
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             100 non-null    int64  
 1   trans_date_trans_time  100 non-null    object 
 2   cc_num                 100 non-null    int64  
 3   merchant               100 non-null    object 
 4   category               100 non-null    object 
 5   amt                    100 non-null    float64
 6   first                  100 non-null    object 
 7   last                   100 non-null    object 
 8   gender                 100 non-null    object 
 9   street                 100 non-null    object 
 10  city                   100 non-null    object 
 11  state                  100 non-null    object 
 12  zip                    100 non-null    int64  
 13  lat                    100 non-null    float64
 14  long                   100 non-null

## 3. Transformaciones aplicadas

Hicimos cuatro transformaciones sobre el DataFrame cargado:

1. **Renombrar columnas al español**, para que el reporte y el análisis sean más fáciles de leer para el grupo.
2. **Eliminar `Unnamed: 0`**, porque es solo un índice que quedó del archivo original y no aporta información.
3. **Rellenar los nulos de `codigo_postal_comerciante` con -1**. Así no perdemos esas filas y el valor -1 deja claro que el dato no existía.
4. **Calcular la `edad`** del cliente al momento de la transacción, restando el año de nacimiento al año de la transacción. Es un cálculo aproximado, porque no tiene en cuenta si la persona ya cumplió años ese año.

In [5]:
# Diccionario con la traducción de las columnas a eEspañol
traduccion_columnas = {
    'trans_date_trans_time': 'fecha_hora_transaccion',
    'cc_num': 'num_tarjeta',
    'merchant': 'establecimiento',
    'category': 'categoria',
    'amt': 'monto',
    'first': 'nombre',
    'last': 'apellido',
    'gender': 'genero',
    'street': 'calle',
    'city': 'ciudad',
    'state': 'estado',
    'zip': 'codigo_postal',
    'lat': 'latitud',
    'long': 'longitud',
    'city_pop': 'poblacion_ciudad',
    'job': 'trabajo',
    'dob': 'fecha_nacimiento',
    'trans_num': 'num_transaccion',
    'unix_time': 'tiempo_unix',
    'merch_lat': 'latitud_comerciante',
    'merch_long': 'longitud_comerciante',
    'is_fraud': 'es_fraude',
    'merch_zipcode': 'codigo_postal_comerciante'
}

# Aplicar el cambio de nombres
df.rename(columns=traduccion_columnas, inplace=True)

In [6]:
# Eliminar la columna innecesaria
df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

In [7]:
# Rellenar valores nulos con -1 para identificarlos fácilmente
df['codigo_postal_comerciante'] = df['codigo_postal_comerciante'].fillna(-1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   fecha_hora_transaccion     100 non-null    object 
 1   num_tarjeta                100 non-null    int64  
 2   establecimiento            100 non-null    object 
 3   categoria                  100 non-null    object 
 4   monto                      100 non-null    float64
 5   nombre                     100 non-null    object 
 6   apellido                   100 non-null    object 
 7   genero                     100 non-null    object 
 8   calle                      100 non-null    object 
 9   ciudad                     100 non-null    object 
 10  estado                     100 non-null    object 
 11  codigo_postal              100 non-null    int64  
 12  latitud                    100 non-null    float64
 13  longitud                   100 non-null    float64


In [8]:
# 1. Extraer solo los primeros 4 caracteres del texto (el año) y convertirlos a número entero
año_trans = df['fecha_hora_transaccion'].str[:4].astype(int)
año_nacimiento = df['fecha_nacimiento'].str[:4].astype(int)

# 2. Calcular la edad simplemente restando los años
df['edad'] = año_trans - año_nacimiento

# Verificar el resultado
print(df[['fecha_hora_transaccion', 'fecha_nacimiento', 'edad']].head())

  fecha_hora_transaccion fecha_nacimiento  edad
0    2019-01-01 00:00:18       1988-03-09    31
1    2019-01-01 00:00:44       1978-06-21    41
2    2019-01-01 00:00:51       1962-01-19    57
3    2019-01-01 00:01:16       1967-01-12    52
4    2019-01-01 00:03:06       1986-03-28    33


## 4. Generar el resumen en `.txt`

Pasamos el DataFrame **ya transformado** a la clase `Reportes`. El archivo queda en `reportes/pointe77/credit-card-transaction.txt` y por eso muestra las columnas en español y la nueva columna `edad`.

In [9]:
from clases_po import Reportes

In [10]:
reportes = Reportes()
texto = reportes.generar_reporte(df)

print(texto)

REPORTE DEL DATAFRAME
Filas: 100
Columnas: 24
Duplicados: 0

Columnas y tipos:
- fecha_hora_transaccion: object
- num_tarjeta: int64
- establecimiento: object
- categoria: object
- monto: float64
- nombre: object
- apellido: object
- genero: object
- calle: object
- ciudad: object
- estado: object
- codigo_postal: int64
- latitud: float64
- longitud: float64
- poblacion_ciudad: int64
- trabajo: object
- fecha_nacimiento: object
- num_transaccion: object
- tiempo_unix: int64
- latitud_comerciante: float64
- longitud_comerciante: float64
- es_fraude: int64
- codigo_postal_comerciante: float64
- edad: int64

Valores nulos por columna:
- fecha_hora_transaccion: 0
- num_tarjeta: 0
- establecimiento: 0
- categoria: 0
- monto: 0
- nombre: 0
- apellido: 0
- genero: 0
- calle: 0
- ciudad: 0
- estado: 0
- codigo_postal: 0
- latitud: 0
- longitud: 0
- poblacion_ciudad: 0
- trabajo: 0
- fecha_nacimiento: 0
- num_transaccion: 0
- tiempo_unix: 0
- latitud_comerciante: 0
- longitud_comerciante: 0
- e

In [11]:
ruta = reportes.guardar_reporte(
df, "reportes/pointe77/credit-card-transaction.txt", 
)
print(f"Reporte guardado en: {ruta}")

Reporte guardado en: reportes/pointe77/credit-card-transaction.txt


## Conclusión

El flujo funciona de punta a punta: la primera clase trae los datos desde Hugging Face, el notebook los transforma y la segunda clase deja el resumen en un archivo de texto. Este mismo mecanismo de carga y reporte es la base que reutilizamos en la actividad 2.